# Module 6: Evals (13 min)

Run automated evaluations against the customer service agent — an **output eval** (is the response good?) and a **trajectory eval** (did the agent follow the right workflow?).

**Prerequisites:** Modules 1-5 completed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Output Evaluation — Is the Response Good?

The `OutputEvaluator` uses LLM-as-a-judge to score agent responses against expected outputs.

In [ ]:
from strands import Agent
from strands_evals import eval_task, Case, Experiment
from strands_evals.evaluators import OutputEvaluator
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests.

Important: Always verify the customer first, then check orders if needed."""


# The @eval_task decorator handles boilerplate
@eval_task()
def get_response():
    return Agent(
        tools=[lookup_customer, get_order_history, process_refund],
        system_prompt=SYSTEM_PROMPT,
        callback_handler=None,
    )


# Define test cases
output_cases = [
    Case[str, str](
        name="order-status-check",
        input="I'm customer C-1001. Where is my USB-C Hub order?",
        expected_output="The USB-C Hub is shipped with tracking TRK-887766, estimated delivery 2025-05-06.",
    ),
    Case[str, str](
        name="delayed-order-empathy",
        input="I'm customer C-1002. My keyboard order is delayed and I'm frustrated!",
        expected_output="Acknowledge frustration, provide order status for the delayed mechanical keyboard with tracking info.",
    ),
    Case[str, str](
        name="unknown-customer",
        input="I'm customer C-9999. What are my orders?",
        expected_output="Inform the customer that no account was found with that ID and ask them to verify.",
    ),
]

# Define the evaluator rubric
output_evaluator = OutputEvaluator(
    rubric="""
    Evaluate the customer service response:
    1. Accuracy — Does it match the expected information?
    2. Tone — Is it professional and empathetic?
    3. Completeness — Does it fully address the customer's concern?

    Score 1.0 if all criteria are met.
    Score 0.5 if partially met.
    Score 0.0 if inadequate or incorrect.
    """,
    include_inputs=True,
)

# Run the experiment
experiment = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator])
reports = experiment.run_evaluations(get_response)

print("=== Output Evaluation Results ===")
reports[0].run_display()

---

## Part 2: Trajectory Evaluation — Did It Follow the Workflow?

The `TrajectoryEvaluator` checks that the agent called tools in the correct order. This validates your steering handlers are working.

In [ ]:
from strands_evals.evaluators import TrajectoryEvaluator
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import TaskOutput


def get_response_with_trajectory(case: Case) -> TaskOutput:
    """Run agent and capture tool trajectory."""
    agent = Agent(
        tools=[lookup_customer, get_order_history, process_refund],
        system_prompt="""You are a customer service agent. When processing refunds, you MUST:
1. First look up the customer
2. Then check their order history
3. Only then process the refund
Always follow this exact order.""",
        callback_handler=None,
    )

    response = agent(case.input)
    trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)

    return TaskOutput(output=str(response), trajectory=trajectory)


# Define cases with expected tool sequences
trajectory_cases = [
    Case[str, str](
        name="refund-workflow",
        input="Customer C-1001 wants a refund for order ORD-5521 ($79.99). Process it now.",
        expected_trajectory=["lookup_customer", "get_order_history", "process_refund"],
    ),
    Case[str, str](
        name="info-lookup-only",
        input="Look up customer C-1001 and tell me their order history.",
        expected_trajectory=["lookup_customer", "get_order_history"],
    ),
    Case[str, str](
        name="customer-lookup-only",
        input="Look up customer C-1002's account info.",
        expected_trajectory=["lookup_customer"],
    ),
]

# Create trajectory evaluator
trajectory_evaluator = TrajectoryEvaluator(
    rubric="""
    Evaluate whether the agent followed the expected tool sequence:
    - The expected tools should appear in order (extra tools in between are OK).
    - Score 1.0 if the expected sequence is followed correctly.
    - Score 0.5 if tools are called but in wrong order.
    - Score 0.0 if expected tools are missing entirely.
    """,
    include_inputs=True,
)

# Give evaluator context about available tools
sample_agent = Agent(tools=[lookup_customer, get_order_history, process_refund])
tool_descriptions = tools_use_extractor.extract_tools_description(sample_agent, is_short=True)
trajectory_evaluator.update_trajectory_description(tool_descriptions)

# Run experiment
experiment = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator])
reports = experiment.run_evaluations(get_response_with_trajectory)

print("=== Trajectory Evaluation Results ===")
reports[0].run_display()

---

## 🎯 Try It Yourself

Add a new test case that checks the agent asks for a customer ID when none is provided:

In [ ]:
# Challenge: Add a case where no customer ID is given
# Expected trajectory should be empty [] — the agent should ask, not guess

# new_case = Case[str, str](
#     name="missing-customer-id",
#     input="I want to return something I bought last week.",
#     expected_trajectory=[],
# )

# Your code here...

---

## What's Next

You've validated the agent works correctly. In **Module 7: Deploy**, you'll package it as a production service using Bedrock AgentCore with persistent memory.